# 10 · Rust/WASM spatial computation

**Question:** When does a compiled spatial kernel help, and what does boundary overhead cost?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Ask Astra/Codex to optimize the Rust kernel without changing its formula, then compare numerical parity and median runtime.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## A small Rust export called from Pyodide
`rust/src/lib.rs` sums an inverse-quadratic distance weight over deterministic synthetic points.
GitHub Actions compiles it to `content/assets/twin_kernel.wasm` before building the site.
In a browser, Pyodide calls JavaScript WebAssembly through its foreign-function interface.
Desktop execution uses Wasmtime when the compiled asset is available. A missing asset is reported explicitly.
The benchmark includes the call boundary but excludes module initialization. It does not establish a general Rust speedup.


In [ ]:
def reference(n,x,y,decay):
    return sum(1/(1+(((i*37%2000)-x)**2+((i*71%2000)-y)**2)/decay**2) for i in range(n))
def vectorized(n,x,y,decay):
    i=np.arange(n); return float(np.sum(1/(1+((i*37%2000-x)**2+(i*71%2000-y)**2)/decay**2)))
asset=Path('assets/twin_kernel.wasm')
wasm_fn=None
if sys.platform=='emscripten':
    import js
    from pyodide.ffi import to_js
    if asset.exists():
        compiled=await js.WebAssembly.compile(to_js(asset.read_bytes()))
        instance=await js.WebAssembly.instantiate(compiled)
        wasm_fn=instance.exports.exposure_sum
elif asset.exists():
    try:
        import wasmtime
        engine=wasmtime.Engine(); store=wasmtime.Store(engine)
        module=wasmtime.Module.from_file(engine,str(asset)); instance=wasmtime.Instance(store,module,[])
        function=instance.exports(store)['exposure_sum']
        wasm_fn=lambda *args:function(store,*args)
    except ImportError:
        print('Desktop Wasmtime is not installed; use the published JupyterLite site.')
print('Rust/WASM loaded' if wasm_fn else 'Rust/WASM NOT loaded. Build it using docs/GUIDE.md; Python comparisons still run.')


In [ ]:
n=20000; args=(n,1000.,900.,200.)
expected=reference(*args)
assert np.isclose(vectorized(*args),expected,rtol=1e-11)
functions={'Python loop':reference,'NumPy':vectorized}
if wasm_fn:
    assert np.isclose(wasm_fn(*args),expected,rtol=1e-11)
    assert wasm_fn(0,0.,0.,100.)==0.
    functions['Rust/WASM']=wasm_fn
timings={}
for name,fn in functions.items():
    fn(*args); samples=[]
    for _ in range(5):
        start=time.perf_counter(); value=fn(*args); samples.append((time.perf_counter()-start)*1000)
    timings[name]=float(np.median(samples))
fig,ax=plt.subplots(); ax.bar(timings.keys(),timings.values(),color=['#087f8c','#f2a541','#536dfe'][:len(timings)])
ax.set(ylabel='Median milliseconds (5 runs)',title=f'Same formula, {n:,} points, this device'); plt.show()
export_json('10_benchmark.json',dict(synthetic=True,wasm_loaded=wasm_fn is not None,n=n,median_ms=timings,reference=expected))


## Extend with Codex or Astra
Batch many queries in one call and measure crossover sizes. Add an R-tree or grid index and validate edge cases. Rust compiles at build time, not inside JupyterLite; a normal native wheel cannot be imported into Pyodide.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
